# Pelatihan Model YOLO11n untuk VNetra

Notebook ini memisahkan tugas berat CPU (download COCO) dan tugas GPU (Merge & Training).


In [ ]:
!pip install -q albumentations
from google.colab import drive
drive.mount('/content/drive')

import os

# --- KONFIGURASI EKSPERIMEN (BISA DIUBAH) ---
EXPERIMENT_ID = 1

DRIVE_BASE_DIR = f'/content/drive/MyDrive/YOLO/eksperimen_{EXPERIMENT_ID}'
INPUT_DIR = f'{DRIVE_BASE_DIR}/input'
OUTPUT_DIR = f'{DRIVE_BASE_DIR}/output'

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import IPython
import PIL
pil_ver = PIL.__version__
print(f"Mengunci versi Pillow ke {pil_ver} untuk mencegah crash C-extension...")
IPython.get_ipython().system(f"pip install ultralytics roboflow pyyaml fiftyone Pillow=={pil_ver}")

import importlib
import site
importlib.reload(site)
importlib.invalidate_caches()

print("Environment siap!")


# --- FASE 1: DOWNLOAD COCO (CPU MODE) ---
Jalankan bagian ini sekali saja. Setelah `coco_subset.zip` tersimpan di Google Drive, Anda tidak perlu menjalankan Fase 1 lagi.


In [ ]:
import os
import shutil

coco_zip_path = f'{INPUT_DIR}/coco_subset.zip'

if os.path.exists(coco_zip_path):
    print(f"✅ COCO Subset sudah ada di {coco_zip_path}!")
    print("ANDA BISA LANGSUNG MELEWATI FASE 1 INI DAN MELANJUTKAN KE FASE 2.")
else:
    print("Mulai mengunduh subset COCO-2017 (Mencegah Catastrophic Forgetting)...")
    import fiftyone as fo
    import fiftyone.zoo as foz
    import time

    coco_classes = ["bicycle", "motorcycle", "bus", "truck", "train", "person", "car"]

    max_retries = 5
    for attempt in range(max_retries):
        try:
            # Mengunduh 30.000 gambar acak yang mengandung kelas-kelas kendaraan/orang.
            # Berkat perbaikan logika di Fase 2, jumlah 30.000 gambar ini akan memberikan
            # jumlah instance (sepeda, motor, bus) yang sangat cukup tanpa dibuang sia-sia!
            print(f"\nSedang mendownload 30.000 gambar COCO... (Percobaan {attempt+1})")
            coco_dataset = foz.load_zoo_dataset(
                "coco-2017", split="train", label_types=["detections"],
                classes=coco_classes, max_samples=30000, num_workers=4
            )
            break
        except Exception as e:
            print(f"Error jaringan saat unduh COCO (Percobaan {attempt+1}/{max_retries}): {e}")
            if attempt == max_retries - 1: raise e
            time.sleep(5)

    coco_export_dir = "/content/coco_subset"
    print("\nMengekspor dataset COCO ke format YOLO...")
    coco_dataset.export(
        export_dir=coco_export_dir, dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth", classes=coco_classes
    )

    print("Sedang men-zip subset COCO...")
    shutil.make_archive(f'{INPUT_DIR}/coco_subset', 'zip', coco_export_dir)
    print(f"✅ COCO Subset berhasil di-zip dan disimpan ke {INPUT_DIR}/coco_subset.zip !")


# --- 🛑 BERHENTI DI SINI 🛑 ---
> **GANTI RUNTIME KE GPU SEKARANG:**
> 1. Klik menu `Runtime` > `Change runtime type`.
> 2. Pilih `T4 GPU`.
> 3. Klik `Save`.
> Setelah me-restart, langsung jalankan sel-sel Fase 2 di bawah.


# --- FASE 2: MERGING & TRAINING (GPU MODE) ---


In [ ]:
import os
import shutil

# Mount ulang dan setup path pasca-restart
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    pass

EXPERIMENT_ID = 1
DRIVE_BASE_DIR = f'/content/drive/MyDrive/YOLO/eksperimen_{EXPERIMENT_ID}'
INPUT_DIR = f'{DRIVE_BASE_DIR}/input'
OUTPUT_DIR = f'{DRIVE_BASE_DIR}/output'

import IPython
IPython.get_ipython().system("pip install ultralytics roboflow pyyaml")

coco_export_dir = "/content/coco_subset"
coco_zip_path = f'{INPUT_DIR}/coco_subset.zip'

if not os.path.exists(coco_export_dir):
    if os.path.exists(coco_zip_path):
        print("Mengekstrak COCO dari Google Drive...")
        os.makedirs(coco_export_dir, exist_ok=True)
        shutil.unpack_archive(coco_zip_path, coco_export_dir)
        print("✅ COCO Subset siap digunakan.")
    else:
        print("❌ ERROR: coco_subset.zip tidak ditemukan. Silakan jalankan Fase 1 terlebih dahulu.")
else:
    print("✅ COCO Subset sudah tersedia di memori.")


## 1. Unduh Dataset Kustom dari Roboflow
Dataset spesialis untuk objek navigasi tunanetra (selain kendaraan/orang yang sudah ada di COCO). Mengunduh langsung dari server Roboflow ke dalam mesin GPU Google Colab.


In [ ]:
# Mengambil API Key secara otomatis dari sistem
import os
from roboflow import Roboflow

roboflow_key = "v8fzCFPTlkekGthVLevM"
rf = Roboflow(api_key=roboflow_key)

# 1. Pothole Dataset (Kelas yang diambil: 'pothole')
dataset_pothole = rf.workspace("yeeun-kim-fyvoj").project("pothole-vhmow").version(18).download("yolov11")

# 2. Tactile Paving Dataset (Kelas yang diambil: straight, turn, 3way, 4way, stop)
dataset_tactile = rf.workspace("raihan-aria").project("paving-tactile-detection").version(4).download("yolov11")

# 3. Open Drain / Selokan Terbuka Dataset (Kelas yang diambil: 'open_drain')
dataset_drain = rf.workspace("chaitanya-kharche").project("drain-overflow").version(2).download("yolov11")

# 4. Puddle / Genangan Air Dataset (Kelas yang diambil: 'puddle')
dataset_puddle = rf.workspace("ambitious-jda7x").project("puddle-zlrsu").version(2).download("yolov11")
dataset_puddle2 = rf.workspace("tt-xsaer").project("fvs").version(4).download("yolov11")

# 5. Pole / Tiang Dataset (Kelas yang diambil: 'pole')
dataset_pole = rf.workspace("ghost-gsj7h").project("utility-pole-aka9k").version(3).download("yolov11")

# 6. Hanging Branch / Ranting Menggantung Dataset (Kelas yang diambil: 'hanging_branch')
dataset_branch = rf.workspace("utem").project("branch-7qne7").version(2).download("yolov11")
dataset_branch3 = rf.workspace("ahmdirfnz").project("branch").version(5).download("yolov11")

# 7. Stairs / Tangga Dataset (Kelas yang diambil: 'stairs_up', 'stairs_down')
dataset_stairs = rf.workspace("jatin-sne2e").project("stairs-zqsvn").version(2).download("yolov11")
dataset_stairs2 = rf.workspace("sovar-sfwov").project("stair-detection-large").version(1).download("yolov11")

# 8. Tree / Pohon Dataset (Kelas yang diambil: 'tree')
dataset_tree = rf.workspace("tree-nqhzs").project("tree-hmf5d").version(1).download("yolov11")

# 9. Crosswalk / Zebra Cross Dataset (Kelas yang diambil: 'crosswalk')
dataset_crosswalk = rf.workspace("wqwdas").project("crosswalk-1elwe").version(1).download("yolov11")

# 10. Fence / Pagar Dataset (Kelas yang diambil: 'fence')
dataset_fence = rf.workspace("ayoub-9grd0").project("fence-detection-bkrx1").version(1).download("yolov11")

# 11. Bench / Bangku Dataset (Kelas yang diambil: 'bench')
dataset_bench1 = rf.workspace("bilab-jhfzv").project("bench-fvynj").version(1).download("yolov11")
dataset_bench2 = rf.workspace("kyonggi-university-r7unh").project("bench-rpf3f").version(1).download("yolov11")
dataset_bench3 = rf.workspace("nvdi-wp3lc").project("bancos-okhbj").version(2).download("yolov11")


## 2. Penggabungan (Merging) Seluruh Dataset
Menyatukan seluruh dataset (11+ Roboflow + 1 COCO) ke dalam folder `vnetra_master_dataset` sambil merekayasa ID Kelas mereka agar berurutan (0-22) secara konsisten dan membatasi jumlah maksimal objek per kelas (Rebalancing).


In [ ]:
import glob
import yaml
import shutil

master_dir = "/content/vnetra_master_dataset"
# KOSONGKAN FOLDER JIKA CELL INI DIJALANKAN ULANG AGAR TIDAK MENUMPUK
if os.path.exists(master_dir):
    shutil.rmtree(master_dir)
for split in ['train', 'valid', 'test']:
    os.makedirs(f"{master_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{master_dir}/{split}/labels", exist_ok=True)

master_classes = [
    'person', 'bicycle', 'car', 'motorcycle', 'train', 
    'bench', 'pothole', 'open_drain', 'puddle', 'pole', 
    'hanging_branch', 'tactile_paving_straight', 'tactile_paving_turn', 
    'tactile_paving_3way', 'tactile_paving_4way', 'tactile_paving_stop', 
    'stairs_up', 'stairs_down', 'crosswalk', 'tree', 'fence'
]
master_class_to_id = {name: i for i, name in enumerate(master_classes)}
coco_classes = ["person", "bicycle", "car", "motorcycle", "bus", "truck", "train"]

def merge_dataset(source_path, class_mapping, is_coco=False, max_samples=None, max_instances_per_class=None, max_instances_per_class_total=None, instance_counter=None):
    if not os.path.exists(source_path): return
    if instance_counter is None: instance_counter = {}
    
    # Baca original classes dari data.yaml (Roboflow) atau dataset.yaml (FiftyOne)
    yaml_path = os.path.join(source_path, 'data.yaml')
    if not os.path.exists(yaml_path):
        yaml_path = os.path.join(source_path, 'dataset.yaml')
    if not os.path.exists(yaml_path): return
    
    with open(yaml_path, 'r') as f:
        yaml_data = yaml.safe_load(f)
        original_classes = yaml_data.get('names', [])
        if isinstance(original_classes, dict):
            original_classes = [original_classes[i] for i in range(len(original_classes))]

    copied_count = 0
    for split in ['train', 'valid', 'test']:
        fo_split = 'val' if split == 'valid' else split
        
        if is_coco and split == 'train':
            if os.path.exists(f"{source_path}/images/val"): fo_split = 'val'
        elif is_coco and split != 'train':
            continue 
            
        img_dir_rf = f"{source_path}/{split}/images"
        img_dir_fo = f"{source_path}/images/{fo_split}"
        
        if os.path.exists(img_dir_rf):
            img_dir = img_dir_rf
            lbl_dir_base = f"{source_path}/{split}/labels"
        elif os.path.exists(img_dir_fo):
            img_dir = img_dir_fo
            lbl_dir_base = f"{source_path}/labels/{fo_split}"
        else:
            continue
            
        import random
        all_images = glob.glob(f"{img_dir}/*")
        random.shuffle(all_images)
        for img_path in all_images:
            if max_samples is not None and copied_count >= max_samples: return
                
            file_name = os.path.basename(img_path)
            lbl_name = file_name.rsplit('.', 1)[0] + '.txt'
            lbl_path = f"{lbl_dir_base}/{lbl_name}"
            if not os.path.exists(lbl_path): continue
                
            new_labels = []
            valid_objects = 0
            with open(lbl_path, 'r') as f: lines = f.readlines()
            
            temp_class_counts = {}
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5: continue
                orig_id = int(parts[0])
                if orig_id >= len(original_classes): continue
                class_name = str(original_classes[orig_id]).lower()
                
                mapped_master_class = None
                for key, val in class_mapping.items():
                    if key.lower() == class_name:
                        mapped_master_class = val
                        break
                if mapped_master_class:
                    temp_class_counts[mapped_master_class] = temp_class_counts.get(mapped_master_class, 0) + 1
                    
            if max_instances_per_class is not None:
                if any(count > max_instances_per_class for count in temp_class_counts.values()): continue
                    
            if max_instances_per_class_total is not None:
                person_cap = 20000 # <-- Variabel khusus untuk kelas perosn (bisa diubah)
                car_cap = 5000  # <-- Variabel khusus untuk kelas car (bisa diubah)
                
                # Kita terima gambar ini JIKA ada MINIMAL SATU kelas di dalamnya yang KUOTANYA BELUM PENUH
                is_needed = False
                for c_name, count in temp_class_counts.items():
                    if c_name == 'person':
                        target_total = person_cap
                    elif c_name == 'car':
                        target_total = car_cap
                    else:
                        target_total = max_instances_per_class_total
                        
                    if instance_counter.get(c_name, 0) < target_total:
                        is_needed = True
                        break
                        
                # BATAS KETAT (STRICT CAP)
                for c_name, count in temp_class_counts.items():
                    if c_name == 'person':
                        strict_cap = person_cap
                    elif c_name == 'car':
                        strict_cap = car_cap
                    else:
                        strict_cap = 2000
                        
                    if instance_counter.get(c_name, 0) >= strict_cap:
                        is_needed = False
                        break
                
                if not is_needed: continue

            
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5: continue
                orig_id = int(parts[0])
                if orig_id >= len(original_classes): continue
                class_name = str(original_classes[orig_id]).lower()
                mapped_master_class = None
                for key, val in class_mapping.items():
                    if key.lower() == class_name:
                        mapped_master_class = val
                        break
                if mapped_master_class:
                    new_id = master_class_to_id[mapped_master_class]
                    new_labels.append(f"{new_id} {' '.join(parts[1:])}\n")
                    valid_objects += 1
            
            if valid_objects > 0:
                for c_name, count in temp_class_counts.items():
                    instance_counter[c_name] = instance_counter.get(c_name, 0) + count
                prefix = source_path.split('/')[-1]
                new_img_name = f"{prefix}_{file_name}"
                new_lbl_name = f"{prefix}_{lbl_name}"
                shutil.copy(img_path, f"{master_dir}/{split}/images/{new_img_name}")
                with open(f"{master_dir}/{split}/labels/{new_lbl_name}", 'w') as f:
                    f.writelines(new_labels)
                    copied_count += 1

print("Memproses COCO Subset (Mencegah Catastrophic Forgetting)...")
coco_instance_counter = {}
custom_coco = {
    "person": "person",
    "bicycle": "bicycle",
    "motorcycle": "motorcycle",
    "car": "car",
    "bus": "car",
    "truck": "car",
    "train": "train"
}
merge_dataset(coco_export_dir, custom_coco, is_coco=True, max_instances_per_class=10, max_instances_per_class_total=4000, instance_counter=coco_instance_counter)
print("Memproses Pothole Dataset...")
merge_dataset(dataset_pothole.location, {"pothole": "pothole"}, max_samples=800)

print("Memproses Tactile Paving Dataset...")
merge_dataset(dataset_tactile.location, {"go": "tactile_paving_straight", "1": "tactile_paving_straight", "0": "tactile_paving_straight", "straight": "tactile_paving_straight", "2": "tactile_paving_turn", "3": "tactile_paving_3way", "4": "tactile_paving_4way", "stop": "tactile_paving_stop"})

# print("Memproses Open Drain Dataset...")
# merge_dataset(dataset_drain.location, {"open drainage-not overflowing-": "open_drain", "open drainage-overflowing-": "open_drain", "drainage overflow-repair-": "open_drain", "open manhole-not overflowing-": "open_drain"})

# print("Memproses Puddle Dataset...")
# merge_dataset(dataset_puddle.location, {"puddle": "puddle"}, max_samples=800)
# merge_dataset(dataset_puddle2.location, {"puddle": "puddle"}, max_samples=400)

print("Memproses Pole Dataset...")
merge_dataset(dataset_pole.location, {"pole": "pole", "pole_including_insulator": "pole"})

print("Memproses Hanging Branch Dataset...")
merge_dataset(dataset_branch.location, {"branch": "hanging_branch", "branches": "hanging_branch", "0": "hanging_branch"})

print("Memproses Stairs Dataset...")
merge_dataset(dataset_stairs.location, {"downstair": "stairs_down", "upstair": "stairs_up", "stairs_up": "stairs_up", "stairs_down": "stairs_down"})
merge_dataset(dataset_stairs2.location, {"downstair": "stairs_down", "upstair": "stairs_up"}, max_samples=3000)

print("Memproses Tree Dataset...")
merge_dataset(dataset_tree.location, {"tree": "tree"}, max_samples=800)

print("Memproses Tambahan Dataset Branch...")
merge_dataset(dataset_branch3.location, {"branches": "hanging_branch"})

# print("Memproses Fence Dataset...")
# merge_dataset(dataset_fence.location, {"fences": "fence"}, max_samples=800)

def merge_dynamic(dataset_loc, target_class, max_samples=None):
    try:
        with open(f"{dataset_loc}/data.yaml", 'r') as f:
            classes = yaml.safe_load(f)['names']
        if isinstance(classes, dict): classes = [classes[i] for i in range(len(classes))]
        cmap = {str(c): target_class for c in classes if str(c).lower() != "null"}
        merge_dataset(dataset_loc, cmap, max_samples=max_samples)
    except Exception as e:
        pass

print("Memproses Crosswalk Dataset...")
merge_dynamic(dataset_crosswalk.location, "crosswalk", max_samples=800)

print("Memproses Tambahan Bench Dataset...")
merge_dynamic(dataset_bench1.location, "bench")
merge_dynamic(dataset_bench2.location, "bench")
merge_dynamic(dataset_bench3.location, "bench")

yaml_content = {
    "path": master_dir,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(master_classes),
    "names": master_classes
}
with open(f"{master_dir}/data.yaml", 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)


### 3.2 Backup Dataset ke Google Drive (Wajib untuk Resume)
Dataset yang sudah digabung akan di-ZIP dan dikirim langsung ke `INPUT_DIR` di Google Drive Anda agar notebook *Resume* dapat mengambilnya kembali jika sesi Colab ini terputus.

In [ ]:
import shutil
import os

print("📦 Membuat arsip ZIP untuk seluruh dataset master...")
dataset_dir = master_dir  
zip_path = f'{INPUT_DIR}/vnetra_master_dataset'

try:
    os.makedirs(INPUT_DIR, exist_ok=True)
    shutil.make_archive(zip_path, 'zip', dataset_dir)
    print(f"✅ Selesai! Dataset master telah diamankan secara permanen ke: {zip_path}.zip")
    print("Di masa depan, Anda bisa menggunakan notebook Resume untuk memanggil dataset ini!")
except Exception as e:
    print(f"❌ Gagal melakukan backup: {e}")
    print("Pastikan Anda telah mengizinkan Google Colab untuk mengakses Google Drive Anda di cell paling atas.")


### 2.1 Jaring Pengaman Rebalancing Data
Mendistribusikan secara adil jumlah gambar (15%) ke dalam keranjang Validation dan Test set, sambil mengembalikan sisa gambar berlebih kembali ke Train set.

In [ ]:
import os
import random
import shutil

print("=== MEMASTIKAN DISTRIBUSI HYBRID VALIDATION & TEST SET (REBALANCING) ===")
train_img_dir = f'{master_dir}/train/images'
train_lbl_dir = f'{master_dir}/train/labels'
valid_img_dir = f'{master_dir}/valid/images'
valid_lbl_dir = f'{master_dir}/valid/labels'
test_img_dir  = f'{master_dir}/test/images'
test_lbl_dir  = f'{master_dir}/test/labels'

for dir_path in [valid_img_dir, valid_lbl_dir, test_img_dir, test_lbl_dir]:
    os.makedirs(dir_path, exist_ok=True)

def get_class_counts(lbl_dir):
    counts = {i: 0 for i in range(len(master_classes))}
    if not os.path.exists(lbl_dir): return counts
    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(lbl_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    if cls_id in counts: counts[cls_id] += 1
    return counts

train_counts = get_class_counts(train_lbl_dir)
valid_counts = get_class_counts(valid_lbl_dir)
test_counts  = get_class_counts(test_lbl_dir)

total_counts = {}
target_valid_test = {}

for cls_id in range(len(master_classes)):
    total = train_counts[cls_id] + valid_counts[cls_id] + test_counts[cls_id]
    total_counts[cls_id] = total
    if total > 0:
        # Ambil 15% dari total, dengan batas maksimum 500 dan minimum 1
        target = max(1, int(0.15 * total))
        target_valid_test[cls_id] = min(target, 500)
    else:
        target_valid_test[cls_id] = 0

def balance_split(target_split_name, target_img_dir, target_lbl_dir, current_counts):
    classes_to_boost = [c for c in range(len(master_classes)) if current_counts[c] < target_valid_test[c]]
    
    if not classes_to_boost:
        print(f"Semua kelas sudah mencapai target hybrid di {target_split_name} Set! Aman.")
        return current_counts

    print(f"Ada kelas yang kurang data di {target_split_name} Set: {classes_to_boost}")
    print(f"Meminjam gambar secara acak dari folder Train untuk {target_split_name}...")
    
    train_labels = [f for f in os.listdir(train_lbl_dir) if f.endswith('.txt')]
    random.shuffle(train_labels)
    
    moved_images = 0
    for lbl_file in train_labels:
        if not classes_to_boost: break
        
        src_lbl = os.path.join(train_lbl_dir, lbl_file)
        contains_needed_class = False
        with open(src_lbl, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            parts = line.strip().split()
            if parts and int(parts[0]) in classes_to_boost:
                contains_needed_class = True
                break
                
        if contains_needed_class:
            dst_lbl = os.path.join(target_lbl_dir, lbl_file)
            img_file_base = os.path.splitext(lbl_file)[0]
            
            src_img, dst_img = None, None
            for ext in ['.jpg', '.jpeg', '.png']:
                temp_src = os.path.join(train_img_dir, img_file_base + ext)
                if os.path.exists(temp_src):
                    src_img = temp_src
                    dst_img = os.path.join(target_img_dir, img_file_base + ext)
                    break
            
            if src_img and os.path.exists(src_img):
                shutil.move(src_img, dst_img)
                shutil.move(src_lbl, dst_lbl)
                moved_images += 1
                
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        c_id = int(parts[0])
                        if c_id in current_counts: 
                            current_counts[c_id] += 1
                            
                classes_to_boost = [c for c in range(len(master_classes)) if current_counts[c] < target_valid_test[c]]

    print(f"Berhasil memindahkan {moved_images} gambar dari Train ke {target_split_name}!")
    return current_counts

print("\n--- 1. HYBRID REBALANCING VALIDATION SET ---")
valid_counts = balance_split("Validation", valid_img_dir, valid_lbl_dir, valid_counts)

print("\n--- 2. HYBRID REBALANCING TEST SET ---")
test_counts = balance_split("Test", test_img_dir, test_lbl_dir, test_counts)

print("\n=== MENGEMBALIKAN KELEBIHAN GAMBAR KE FOLDER TRAIN (STRICT CAPPING) ===")
train_counts = get_class_counts(train_lbl_dir)

def return_excess_to_train(source_name, source_img_dir, source_lbl_dir, current_counts):
    # DIBUANG: and train_counts[c] < current_counts[c]
    classes_to_reduce = [c for c in range(len(master_classes)) if current_counts[c] > target_valid_test[c]]
    
    if not classes_to_reduce:
        return current_counts
        
    print(f"Mengembalikan kelebihan data dari {source_name} ke Train untuk kelas: {classes_to_reduce}")
    
    labels_list = [f for f in os.listdir(source_lbl_dir) if f.endswith('.txt')]
    random.shuffle(labels_list)
    moved_back = 0
    
    for lbl_file in labels_list:
        if not classes_to_reduce: break
        
        src_lbl = os.path.join(source_lbl_dir, lbl_file)
        contains_excess_class = False
        with open(src_lbl, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            parts = line.strip().split()
            if parts and int(parts[0]) in classes_to_reduce:
                contains_excess_class = True
                break
                
        if contains_excess_class:
            dst_lbl = os.path.join(train_lbl_dir, lbl_file)
            img_file_base = os.path.splitext(lbl_file)[0]
            
            src_img, dst_img = None, None
            for ext in ['.jpg', '.jpeg', '.png']:
                temp_src = os.path.join(source_img_dir, img_file_base + ext)
                if os.path.exists(temp_src):
                    src_img = temp_src
                    dst_img = os.path.join(train_img_dir, img_file_base + ext)
                    break
            
            if src_img and os.path.exists(src_img):
                can_move = True
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        c_id = int(parts[0])
                        if current_counts[c_id] <= target_valid_test[c_id]:
                            can_move = False
                            break
                
                if can_move:
                    shutil.move(src_img, dst_img)
                    shutil.move(src_lbl, dst_lbl)
                    moved_back += 1
                    
                    for line in lines:
                        parts = line.strip().split()
                        if parts:
                            c_id = int(parts[0])
                            current_counts[c_id] -= 1
                            train_counts[c_id] += 1
                                
                    classes_to_reduce = [c for c in range(len(master_classes)) if current_counts[c] > target_valid_test[c]]
                    
    print(f"Berhasil mengembalikan {moved_back} gambar dari {source_name} ke Train!")
    return current_counts

valid_counts = return_excess_to_train("Validation", valid_img_dir, valid_lbl_dir, valid_counts)
test_counts = return_excess_to_train("Test", test_img_dir, test_lbl_dir, test_counts)
print("\nDistribusi Hybrid Selesai! Model akan aman dari Catastrophic Forgetting untuk kelas kecil.")



### 2.2 Laporan Akhir Proporsi Dataset
Mencetak tabel distribusi dari keseluruhan dataset untuk verifikasi manual.

In [ ]:
import pandas as pd
import os

def count_images(directory):
    if not os.path.exists(directory): return 0
    return len([f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))])

train_count = count_images(f'{master_dir}/train/images')
valid_count = count_images(f'{master_dir}/valid/images')
test_count  = count_images(f'{master_dir}/test/images')
total_images = train_count + valid_count + test_count

print("=== Statistik Keseluruhan ===")
print(f"Total Lembar Gambar (All) : {total_images} gambar")
print(f"Total Gambar Training     : {train_count} gambar")
print(f"Total Gambar Validasi     : {valid_count} gambar")
print(f"Total Gambar Testing      : {test_count} gambar")
print("=============================")
print("")

def count_instances_per_class(label_dir, num_classes):
    counts = {i: 0 for i in range(num_classes)}
    if not os.path.exists(label_dir): return counts
    for lbl_file in os.listdir(label_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(label_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts: counts[int(parts[0])] += 1
    return counts

train_cls = count_instances_per_class(f'{master_dir}/train/labels', len(master_classes))
valid_cls = count_instances_per_class(f'{master_dir}/valid/labels', len(master_classes))
test_cls  = count_instances_per_class(f'{master_dir}/test/labels', len(master_classes))

data_report = []
total_train = 0
total_valid = 0
total_test = 0
global_total = 0

for i, cls_name in enumerate(master_classes):
    t_train = train_cls[i]
    t_valid = valid_cls[i]
    t_test = test_cls[i]
    t_total = t_train + t_valid + t_test
    
    total_train += t_train
    total_valid += t_valid
    total_test += t_test
    global_total += t_total
    
    data_report.append({
        'ID': i, 
        'Kelas': cls_name, 
        'Train (Inst)': t_train, 
        'Valid (Inst)': t_valid, 
        'Test (Inst)': t_test,
        'Total Instance': t_total
    })

data_report.append({
    'ID': '-', 
    'Kelas': 'TOTAL KESELURUHAN', 
    'Train (Inst)': total_train, 
    'Valid (Inst)': total_valid, 
    'Test (Inst)': total_test,
    'Total Instance': global_total
})

df_report = pd.DataFrame(data_report)
display(df_report)


## 3. Training YOLO11n dengan Augmentasi OV2640
Melakukan proses pelatihan model YOLO11n dengan pengaturan hyperparameter khusus (Mosaic, rotasi, fluktuasi warna) untuk mensimulasikan tangkapan kamera OV2640 yang dipasang di dada pengguna tunanetra.


In [ ]:
from ultralytics import YOLO

# Memuat arsitektur dasar YOLO11 versi nano (Paling ringan dan cepat untuk mobile)
model = YOLO('yolo11n.pt')

results = model.train(
    # --- KONFIGURASI DATA & PERANGKAT ---
    data=f"{master_dir}/data.yaml", # Path menuju dataset yang sudah digabung
    epochs=300,                     # Maksimal putaran training (300 sudah lebih dari cukup)
    time=11.0,                      # Otomatis Berhenti & Save dengan aman setelah 11 jam (Mencegah Colab mendadak mati)
    patience=50,                    # Jika dalam 50 epoch akurasi tidak naik, hentikan training lebih awal (Early Stopping)
    imgsz=640,                      # Resolusi standar YOLO (kamera OV2640 akan di-resize ke ukuran ini)
    batch=32,                       # Memproses 32 gambar sekaligus (menyesuaikan kapasitas RAM GPU T4 Colab)
    device=0,                       # Menggunakan GPU ke-0 (Wajib menggunakan GPU untuk YOLO)
    workers=4,                      # Menggunakan 4 core CPU untuk memuat gambar ke GPU lebih cepat
    seed=42,                        # Angka acak tetap agar hasil training bisa direproduksi/konsisten
    
    # --- PENYIMPANAN LOG & GRAFIK ---
    project='vnetra_training',      # Nama folder utama penyimpanan hasil
    name='yolo11n_custom',          # Nama sub-folder spesifik untuk eksperimen ini
    exist_ok=True,                  # Menimpa folder jika sudah ada (mencegah penumpukan folder eksperimen)
    save_period=10,                 # Menyimpan file bobot cadangan setiap 10 putaran
    
    # --- STRATEGI PEMBELAJARAN (LEARNING) ---
    freeze=5,                       # Membekukan (tidak melatih ulang) 5 layer awal yang sudah mahir mendeteksi tepi benda (menghemat waktu)
    lr0=0.002,                      # Kecepatan belajar awal (tidak terlalu besar agar tidak 'nyasar', tidak terlalu kecil agar tidak lambat)
    cos_lr=True,                    # Menurunkan kecepatan belajar secara perlahan membentuk kurva kosinus (memuluskan akurasi di akhir)
    warmup_epochs=1.0,              # Pemanasan 1 epoch pertama dengan kecepatan sangat rendah agar model tidak kaget
    
    # --- AUGMENTASI KHUSUS VNETRA (OV2640 CAMERA SIMULATION) ---
    mosaic=1.0,                     # Menggabungkan 4 gambar jadi 1, melatih model mendeteksi objek kecil dalam satu frame
    degrees=15.0,                   # Memutar gambar hingga 15 derajat (Kamera di dada tunanetra seringkali miring saat berjalan)
    fliplr=0.0,                     # DIMATIKAN! Jangan membalik gambar kiri-kanan, karena arah Tactile Paving (belok kiri vs kanan) bisa tertukar
    scale=0.3,                      # Men-zoom in/out gambar sebesar 30% (Simulasi objek yang kadang dekat atau jauh dari kamera)
    
    # Simulasi kualitas gambar buruk dari kamera OV2640 (warna pudar, gelap, dll)
    hsv_h=0.015,                    # Fluktuasi hue (warna dasar)
    hsv_s=0.9,                      # Fluktuasi saturation EKSTREM (90%): Simulasi warna sangat pucat (noise malam) atau sangat mencolok
    hsv_v=0.8,                      # Fluktuasi value EKSTREM (80%): Simulasikan Siang terik & Malam hari gelap gulita
    erasing=0.3,                    # Menghapus/menutup sebagian kecil gambar secara acak (Simulasi objek tertutup rintangan)
)


In [ ]:
# Pindahkan bobot dan grafik hasil training ke Output Dir
import shutil
import os

best_pt_path = 'vnetra_training/yolo11n_custom/weights/best.pt'
if os.path.exists(best_pt_path):
    shutil.copy(best_pt_path, f'{OUTPUT_DIR}/best_yolo11n.pt')
    print("✅ Model Asli (.pt) berhasil disimpan ke Google Drive (Folder Output)!")

source_graph_dir = 'vnetra_training/yolo11n_custom'
target_graph_dir = f'{OUTPUT_DIR}/training_graphs'
if os.path.exists(source_graph_dir):
    if os.path.exists(target_graph_dir): shutil.rmtree(target_graph_dir)
    shutil.copytree(source_graph_dir, target_graph_dir)


## 4. Export ke TensorFlow Lite (TFLite)
Mengekspor bobot model menjadi format `.tflite` dalam bentuk kuantisasi **FP16** (Half Precision) yang sangat efisien dan kompatibel untuk *GPU Delegation* di smartphone Android.


In [ ]:
export_fp16 = model.export(format="tflite", half=True, optimize=True)
import shutil
shutil.copy(export_fp16, f'{OUTPUT_DIR}/best_fp16.tflite')
print('Model FP16 berhasil disimpan ke Google Drive!')


## 5. Validasi Kuantisasi (Benchmarking Skripsi)
Menguji kembali model pada Test Set untuk melihat seberapa jauh penurunan akurasi (mAP) akibat proses kompresi FP16 dibanding model aslinya.

In [ ]:
import gc
gc.collect()

print("\n=== EVALUASI MODEL ASLI (.pt) PADA TEST SET ===")
val_pt = model.val(data=f"{master_dir}/data.yaml", split='test')
map_pt = val_pt.box.map50

print("\n=== EVALUASI MODEL FP16 (.tflite) PADA TEST SET ===")
model_fp16 = YOLO(export_fp16, task='detect')
val_fp16 = model_fp16.val(data=f"{master_dir}/data.yaml", split='test')
map_fp16 = val_fp16.box.map50

print("\n=========================================")
print("KESIMPULAN PERBANDINGAN mAP@50 PADA DATASET TEST:")
print(f"Original (.pt)   : {map_pt:.4f}")
print(f"FP16 (.tflite)   : {map_fp16:.4f}")
print("=========================================")


## 6. Pengujian Visualisasi Langsung (Predict)
Mengambil satu gambar tes secara acak dan menampilkan prediksi kotak deteksi dari model asli (.pt) vs model terkompresi (.tflite) agar Anda bisa meletakkannya di Laporan Skripsi.

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
import glob

test_images = glob.glob(f"{master_dir}/test/images/*.jpg")
if test_images:
    test_img = random.choice(test_images)
    res_pt = model.predict(source=test_img, imgsz=640)
    img_pt = res_pt[0].plot()
    
    res_fp16 = model_fp16.predict(source=test_img, imgsz=640)
    img_fp16 = res_fp16[0].plot()
    
    fig, ax = plt.subplots(1, 2, figsize=(15, 7))
    ax[0].imshow(cv2.cvtColor(img_pt, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Prediksi Model Asli (.pt)")
    ax[0].axis("off")
    ax[1].imshow(cv2.cvtColor(img_fp16, cv2.COLOR_BGR2RGB))
    ax[1].set_title("Prediksi Model FP16 (.tflite)")
    ax[1].axis("off")
    plt.show()


## 7. Visualisasi Grafik Hasil Training
Menampilkan grafik metrik akurasi (*mAP*, *Loss*) dan *Confusion Matrix* yang telah digenerasi oleh YOLO menggunakan `matplotlib` untuk keperluan laporan skripsi.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

base_path = '/content/runs/detect/vnetra_training/yolo11n_custom/'
results_path = os.path.join(base_path, 'results.csv')

if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()
    sns.set_theme(style='whitegrid', palette='deep')
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    fig.suptitle('VNetra - YOLO11n Training Performance Dashboard', fontsize=26, fontweight='bold', y=0.96)
    
    sns.lineplot(data=df, x='epoch', y='train/box_loss', ax=axes[0,0], label='Train Box Loss', linewidth=3)
    sns.lineplot(data=df, x='epoch', y='val/box_loss', ax=axes[0,0], label='Val Box Loss', linewidth=3, linestyle='--')
    axes[0,0].set_title('Box Loss Convergence', fontsize=18, fontweight='bold')
    
    sns.lineplot(data=df, x='epoch', y='train/cls_loss', ax=axes[0,1], label='Train Class Loss', linewidth=3)
    sns.lineplot(data=df, x='epoch', y='val/cls_loss', ax=axes[0,1], label='Val Class Loss', linewidth=3, linestyle='--')
    axes[0,1].set_title('Classification Loss Convergence', fontsize=18, fontweight='bold')
    
    sns.lineplot(data=df, x='epoch', y='metrics/mAP50(B)', ax=axes[1,0], label='mAP@50', linewidth=3)
    sns.lineplot(data=df, x='epoch', y='metrics/mAP50-95(B)', ax=axes[1,0], label='mAP@50-95', linewidth=3, linestyle='-.')
    axes[1,0].set_title('Mean Average Precision (mAP)', fontsize=18, fontweight='bold')
    
    sns.lineplot(data=df, x='epoch', y='metrics/precision(B)', ax=axes[1,1], label='Precision', linewidth=3)
    sns.lineplot(data=df, x='epoch', y='metrics/recall(B)', ax=axes[1,1], label='Recall', linewidth=3, linestyle=':')
    axes[1,1].set_title('Precision & Recall Trends', fontsize=18, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

def display_result(image_path, width=None):
    if os.path.exists(image_path):
        if width: display(Image(filename=image_path, width=width))
        else: display(Image(filename=image_path))

print('\n=== CONFUSION MATRIX ===')
display_result(os.path.join(base_path, 'confusion_matrix_normalized.png'), width=1200)

print('\n=== KURVA F1-SCORE ===')
display_result(os.path.join(base_path, 'F1_curve.png'), width=1200)

print('\n=== AUGMENTASI MOSAIC ===')
display_result(os.path.join(base_path, 'train_batch0.jpg'), width=1200)

print('\n=== PREDIKSI PADA VALIDATION SET ===')
display_result(os.path.join(base_path, 'val_batch0_pred.jpg'), width=1200)


### 💾 5. Auto-Backup Hasil Training ke Google Drive
Seluruh metrik, grafik, dan bobot (`.pt`) akan otomatis dikompres menjadi file ZIP dan dikirim ke Google Drive agar aman dari disk *reset* Colab.

In [ ]:
import os
import shutil

print(f"\nMenge-ZIP dan membackup seluruh hasil training ke {OUTPUT_DIR}...")
# Menyimpan langsung ke variabel OUTPUT_DIR yang sudah di-set di awal notebook
shutil.make_archive(f"{OUTPUT_DIR}/vnetra_training_results", 'zip', "/content/runs/detect/vnetra_training")

print(f"\nMenyalin file model (.pt) secara langsung (tanpa di-zip) ke {OUTPUT_DIR}...")
weights_dir = "/content/runs/detect/vnetra_training/yolo11n_custom/weights"
if os.path.exists(weights_dir):
    for pt_file in ["best.pt", "last.pt"]:
        src_pt = f"{weights_dir}/{pt_file}"
        dst_pt = f"{OUTPUT_DIR}/{pt_file}"
        if os.path.exists(src_pt):
            shutil.copy2(src_pt, dst_pt)
            print(f"✔️ Berhasil menyalin {pt_file}")
        else:
            print(f"⚠️ Peringatan: {pt_file} tidak ditemukan di {weights_dir}")

print(f"\n✅ BERHASIL! Seluruh grafik & log aman di ZIP, dan file Model bisa langsung diakses di:")
print(f"📁 {OUTPUT_DIR}/")
